# Polars

A refresher on **Polars** — a DataFrame library for tabular data, written in Rust, built on
Apache Arrow, and designed around a **lazy, expression-based** query engine. Think of it as
"pandas, redone with a query optimizer and real parallelism." For the SQL-first take on the
same columnar/Arrow world see [[duckdb]]; for the classic in-memory workflow see
[[numpy-pandas-scipy]].

**Domain:** Data Analysis & Research  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** Polars is a DataFrame library with a pandas-like surface but a fundamentally
different engine. Data is stored column-wise in **Apache Arrow** memory; operations are
expressed as **expressions** (`pl.col("x") * 2`) that the engine can analyze, reorder, and run
in parallel across all your cores. It has two front ends: an **eager** API (runs immediately,
like pandas) and a **lazy** API (builds a query plan that's optimized before execution).

**The problem it solves.** pandas is single-threaded, eager, and its object/index model wastes
memory and surprises people (the index, silent upcasts to `object`, `SettingWithCopyWarning`).
On a few-hundred-MB CSV with a couple of joins and group-bys, pandas pins one core and may
blow past RAM. Polars runs the same workload **multi-threaded**, with a query optimizer that
pushes filters down to the scan and reads only the columns you actually use — routinely
5–30× faster with a fraction of the memory, and it can **stream** datasets larger than RAM.

**When to reach for it.** Medium-to-large tabular ETL and analytics (roughly 100 MB – 100+ GB),
anything where pandas is too slow or won't fit in memory, and new pipelines where you want a
clean, predictable API. **When not to:** tiny data where speed is irrelevant and pandas
muscle-memory wins; deep integration with the SciPy/scikit-learn/statsmodels stack that speaks
pandas natively (convert with `.to_pandas()` at the boundary); or when a teammate needs SQL —
reach for [[duckdb]], which interoperates with Polars over Arrow at zero copy.

## 2. Mental Model

**You don't tell Polars *how* to compute row by row — you describe *what* each output column is
as an expression, and the engine figures out the fastest way to produce all of them at once,
in parallel.**

The unit of thought is the **expression**, not the cell. `pl.col("temp").mean().over("city")`
is a recipe — "the mean temperature within each city" — not an immediate value. You hand a bag
of these recipes to a *context* (`select`, `with_columns`, `group_by(...).agg`, `filter`) and
Polars executes them together, dividing the work across cores.

The second half of the model is **lazy vs eager**. In lazy mode you chain operations on a
`LazyFrame`; nothing runs until you call `.collect()`. In between, an optimizer rewrites the
plan: **predicate pushdown** (apply filters during the file scan), **projection pushdown**
(read only the needed columns), common-subexpression elimination, and more. Same code, but the
engine never materializes data it's about to throw away.

If pandas is an interpreter executing each statement immediately, Polars is a **compiler**: it
sees the whole query, optimizes it, then runs.

## 3. Key Concepts

- **Expression (`pl.col`, `pl.lit`).** A lazy, composable description of a column computation.
  Expressions are the heart of Polars — you build them up (`pl.col("a").fill_null(0).sum()`) and
  evaluate them inside a context.
- **Contexts: `select`, `with_columns`, `group_by().agg`, `filter`.** Where expressions run.
  `select` returns *only* the chosen columns; `with_columns` *adds/overwrites* columns and
  keeps the rest; `agg` evaluates expressions per group; `filter` keeps rows.
- **Eager vs lazy.** `pl.DataFrame` / `pl.read_*` execute now. `pl.LazyFrame` / `pl.scan_*`
  build a plan run by `.collect()`. Inspect the plan with `.explain()`. **Prefer lazy** for
  anything non-trivial — it's where the optimizer earns its keep.
- **`scan_*` vs `read_*`.** `read_csv`/`read_parquet` load the whole file eagerly. `scan_csv`/
  `scan_parquet` return a LazyFrame so filters and column selection are pushed into the scan —
  you read only what you need.
- **Window functions: `.over(...)`.** Compute an aggregation *per group* but broadcast it back
  to every row (group mean as a new column, rank within group) without collapsing the frame.
- **Streaming / out-of-core.** `.collect(engine="streaming")` processes data in chunks so
  queries can exceed RAM. Not every operation streams, but scans, filters, joins, and
  group-bys generally do.
- **No index.** Unlike pandas there's no row index — rows are positional, and "index-like"
  work is done with explicit columns and joins. Less magic, fewer surprises.
- **Strict, Arrow-backed dtypes.** Columns have real types (`Int64`, `Float64`, `Utf8`,
  `Categorical`, `Datetime`, `List`...). No silent `object` fallback; nulls are first-class and
  distinct from NaN.

## 4. Setup

Polars is a single wheel with no Python dependencies — the Rust engine is bundled. CPU-only,
installs in seconds. (`pip install 'polars[all]'` adds optional extras like fsspec, pandas
interop, and Excel I/O.)

In [ ]:
# %pip install polars
import polars as pl

print("polars", pl.__version__)
print("threads", pl.thread_pool_size())  # Polars parallelizes across these by default

## 5. Worked Examples

A tiny weather table is enough to show the whole idea: expressions, contexts, group-bys,
window functions, and the lazy optimizer. Everything here is CPU-only and runs in a fresh
kernel.

### Example 1 — Expressions and contexts (the eager API)

In [ ]:
df = pl.DataFrame(
    {
        "city": ["NYC", "NYC", "LA", "LA", "SF", "SF"],
        "month": ["Jan", "Feb", "Jan", "Feb", "Jan", "Feb"],
        "temp_f": [32, 35, 60, 62, 55, 58],
        "rain_in": [3.2, 3.1, 1.0, 0.8, 4.1, 3.9],
    }
)
print(df)

# A context (`group_by().agg`) evaluates a bag of expressions per group, in parallel.
summary = (
    df.filter(pl.col("temp_f") > 40)               # keep warm rows
      .group_by("city")
      .agg(
          pl.col("temp_f").mean().alias("avg_temp"),
          pl.col("rain_in").sum().alias("total_rain"),
          pl.len().alias("n_rows"),
      )
      .sort("avg_temp", descending=True)
)
print(summary)

Note what's *not* here: no index, no `inplace=`, no `reset_index()`. Each aggregation is an
expression; the engine runs them together. `pl.len()` counts rows per group (the Polars
equivalent of `size()`).

### Example 2 — Window functions with `.over()`

Window expressions compute a per-group aggregate but keep every row — ideal for "value minus
its group mean" or "rank within group" without a join back.

In [ ]:
enriched = df.with_columns(
    pl.col("temp_f").mean().over("city").alias("city_avg_temp"),
    (pl.col("temp_f") - pl.col("temp_f").mean().over("city")).alias("temp_vs_city"),
    pl.col("rain_in").rank("dense", descending=True).over("city").alias("rain_rank_in_city"),
    # Convert to Celsius while we're here — expressions chain freely.
    ((pl.col("temp_f") - 32) * 5 / 9).round(1).alias("temp_c"),
)
print(enriched)

`with_columns` *adds* columns and keeps the original ones (contrast `select`, which returns
only what you ask for). `.over("city")` is the key move: the mean is computed per city but
broadcast back to all of that city's rows, so `temp_vs_city` and the in-city rank line up
row-for-row with the source.

### Example 3 — The lazy API and the query optimizer

The payoff of lazy mode is visible in the optimized plan. We write a CSV, `scan_csv` it (lazy),
filter and aggregate, and inspect what the engine *actually* decides to run.

In [ ]:
import tempfile, os

# Write a small CSV to disk so we can scan it lazily (no network).
path = os.path.join(tempfile.gettempdir(), "weather.csv")
df.write_csv(path)

lazy = (
    pl.scan_csv(path)                       # LazyFrame: nothing read yet
      .filter(pl.col("temp_f") > 40)
      .group_by("city")
      .agg(pl.col("rain_in").sum().alias("total_rain"))
)

print("=== Optimized physical plan (read bottom-up) ===")
print(lazy.explain())                       # note: FILTER and a projected column set pushed into the scan

print("\n=== Result (only now does it execute) ===")
print(lazy.collect())

Read `explain()` from the bottom up. The optimizer pushed the `temp_f > 40` **predicate** down
into the CSV scan and restricted the **projection** to just the columns the query needs — so on
a real file it would never read the rows or columns it's about to discard. That's the same code
you'd write eagerly, but the engine planned it first. For data larger than RAM, swap
`.collect()` for `.collect(engine="streaming")`.

### Optional — read a real CSV over the network (gated)

Kept behind an env check so the notebook always executes offline; it still shows the call shape.

In [ ]:
if os.getenv("FETCH_CSV"):
    url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv"
    iris = pl.read_csv(url)
    print(
        iris.group_by("species").agg(
            pl.col("sepal_length").mean().round(2).alias("avg_sepal_len"),
            pl.len().alias("n"),
        )
    )
else:
    print("Set FETCH_CSV=1 to download and aggregate the iris dataset.")
    print("Call shape:")
    print("  pl.read_csv(url).group_by('species').agg(pl.col('sepal_length').mean())")
    print("  # or lazily over a path/glob:")
    print("  pl.scan_csv('data/*.csv').filter(...).group_by(...).collect(engine='streaming')")

## 6. Gotchas & Pitfalls

- **There is no index.** Habits like `df.loc[...]`, `set_index`, and index-aligned arithmetic
  don't exist. Select rows with `filter`, align data with explicit `join`s. This is a feature —
  fewer hidden alignment bugs — but it trips up pandas muscle memory.
- **`with_columns` vs `select`.** `select` returns *only* the listed columns; `with_columns`
  *adds* them to the existing frame. Reaching for `select` when you meant `with_columns` is the
  classic "where did my other columns go?" surprise.
- **null is not NaN.** Polars distinguishes missing (`null`) from floating-point `NaN`. Use
  `.fill_null()` / `.drop_nulls()` / `.is_null()` for missingness; `NaN` only arises from float
  math (`0/0`) and is handled separately (`.fill_nan`, `.is_nan`).
- **Don't loop over rows.** `.iter_rows()` / `apply` with Python functions throws away every
  advantage — you're back to single-threaded Python. Express the logic as column expressions
  (`when/then/otherwise`, arithmetic, string/`list` namespaces) so the Rust engine runs it.
- **Eager `read_csv` reads everything.** If you only need a few columns or rows from a big file,
  use `scan_csv(...).select(...).filter(...).collect()` so pushdown reads less off disk.
  Eager `read_*` defeats the optimizer.
- **Streaming isn't universal.** `engine="streaming"` handles scans/filters/joins/group-bys, but
  some operations still materialize. If a streaming query OOMs, check which node forced a full
  collect (it's evolving fast across versions).
- **pandas interop costs a copy sometimes.** `.to_pandas()` is zero-copy for Arrow-friendly
  dtypes but copies for others; converting back and forth in a hot loop erases the speedup.
  Convert once at the boundary.
- **The API moves fast.** Polars pre/post-1.0 renamed things (`pl.count` → `pl.len`,
  `groupby` → `group_by`, `with_column` → `with_columns`). Pin a version and check the
  changelog before upgrading across minor releases.

## 7. When to Use vs Alternatives

| Tool | Use when | Trade-off |
|---|---|---|
| **Polars** (this notebook) | Medium–large tabular ETL/analytics; pandas too slow or won't fit in RAM; new pipelines wanting a clean API | Smaller ecosystem than pandas; some SciPy/ML libs need a `.to_pandas()` boundary; API still evolving |
| **pandas** ([[numpy-pandas-scipy]]) | Small data, exploratory work, deep integration with scikit-learn/statsmodels/matplotlib | Single-threaded, eager, heavier memory; index/dtype foot-guns |
| **DuckDB** ([[duckdb]]) | You'd rather write SQL; joins/aggregations over Parquet; embedded analytical DB | SQL not Python expressions; pairs *with* Polars over Arrow zero-copy rather than competing |
| **Dask / Spark** | Truly distributed, multi-machine, terabyte+ workloads | Cluster overhead and complexity; on a single big box Polars streaming is usually simpler and faster |
| **NumPy** ([[numpy-pandas-scipy]]) | Pure numerical arrays, no heterogeneous columns or grouping | Not a DataFrame — no named columns, joins, or group-by ergonomics |

**Rule of thumb:** single-machine tabular work where pandas hurts → Polars. Want SQL or to
query Parquet directly → DuckDB (and hand results to Polars over Arrow). Genuinely
multi-machine → Spark/Dask. Tiny data you already know in pandas → just use pandas.

## 8. Resources

- **Polars User Guide** (concepts, expressions, lazy API, streaming — start here):
  https://docs.pola.rs/
- **Python API reference** (every method, searchable):
  https://docs.pola.rs/api/python/stable/reference/index.html
- **"Coming from pandas" migration guide** (the mental-model translation table):
  https://docs.pola.rs/user-guide/migration/pandas/
- **Modern Polars** by Kevin Heavey (side-by-side pandas↔Polars idioms, free online book):
  https://kevinheavey.github.io/modern-polars/
- **Polars GitHub** (source, issues, release notes — the API moves fast):
  https://github.com/pola-rs/polars

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def optimize(available, ops):
    """Push filters into the scan and work out which stored columns must be read."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE